# 01 Cleaning: F1 2026 season

Source: [F1DB](https://github.com/f1db/f1db) release `v2026.15.1`
(CC BY 4.0), published 9/27 right after the Azerbaijan GP.
Raw CSVs are in `../data/raw/`. This notebook only cleans and saves
tables to `../data/clean/`. No charts here; those live in 02 and 03.

What we keep: the 15 races run so far (rounds 1 to 15).

In [ ]:
import pandas as pd
from pathlib import Path

RAW = Path("../data/raw")
CLEAN = Path("../data/clean")
CLEAN.mkdir(exist_ok=True)
YEAR, LAST_ROUND = 2026, 15

def raw(name):
    return pd.read_csv(RAW / f"f1db-{name}.csv", low_memory=False)

## 1. Load and filter to 2026

F1DB covers every season since 1950. We only want this year.

In [2]:
results = raw("races-race-results")
pits = raw("races-pit-stops")
d_stand = raw("races-driver-standings")
t_stand = raw("races-constructor-standings")
races = raw("races")

def this_year(df):
    return df[(df["year"] == YEAR)
              & (df["round"] <= LAST_ROUND)].copy()

results, pits = this_year(results), this_year(pits)
d_stand, t_stand = this_year(d_stand), this_year(t_stand)
cal = races[races["year"] == YEAR].copy()  # all 23, for the map

for n, df in [("results", results), ("pit stops", pits),
              ("driver standings", d_stand),
              ("team standings", t_stand), ("calendar", cal)]:
    print(f"{n:17} {df.shape}")

results           (330, 34)
pit stops         (469, 15)
driver standings  (333, 10)
team standings    (165, 11)
calendar          (23, 47)


## 2. Audit missing values

Most blanks here are not errors, they mean something:

- `points` blank = the driver scored nothing
- `reasonRetired` blank = the car finished
- `positionNumber` blank = no classified finish (DNF, DNS, NC)

In [3]:
cols = ["positionNumber", "positionText", "gridPositionNumber",
        "points", "reasonRetired", "pitStops"]
results[cols].isna().mean().round(2).to_frame("share missing")

,share missing
positionNumber,0.20
positionText,0.00
gridPositionNumber,0.03
points,0.55
reasonRetired,0.78
pitStops,0.00


In [4]:
results["positionText"].where(
    results["positionNumber"].isna()).value_counts()

positionText
DNF    57
DNS     7
NC      2
Name: count, dtype: int64

## 3. Readable names

Swap the ids (`kimi-antonelli`, `red-bull`) for display names.
Hadjar missed rounds 12 to 14. Lawson moved up to Red Bull and
Tsunoda filled the Racing Bulls seat, then everyone went back for
round 15. So a driver's team is taken per race, not per season.

In [5]:
drivers = raw("drivers").set_index("id")
teams = raw("constructors").set_index("id")
gps = raw("grands-prix").set_index("id")
circuits = raw("circuits").set_index("id")

def add_names(df):
    df["driver"] = df["driverId"].map(drivers["name"])
    df["code"] = df["driverId"].map(drivers["abbreviation"])
    if "constructorId" in df:
        df["team"] = df["constructorId"].map(teams["name"])
    return df

cal["gp"] = cal["grandPrixId"].map(gps["name"])
gp_of = cal.set_index("round")["gp"]

## 4. Race results table

One row per driver per race. `status` collapses the position text
into Finished / DNF / DNS / NC. `gained` = grid minus finish, so a
positive number means the driver moved forward. Pit lane starts
come through as grid 0; we treat those as starting last (P22).

In [6]:
res = add_names(results)
res["gp"] = res["round"].map(gp_of)
res["points"] = res["points"].fillna(0)

txt = res["positionText"]
res["status"] = "Finished"
res.loc[txt.isin(["DNF", "DNS", "NC"]), "status"] = txt

res["grid"] = res["gridPositionNumber"]
res.loc[res["grid"].isin([0]) | res["grid"].isna(), "grid"] = 22
res["finish"] = res["positionNumber"]
res["gained"] = res["grid"] - res["finish"]

res = res[["round", "gp", "driver", "code", "team", "grid",
           "finish", "status", "reasonRetired", "points",
           "gained", "pitStops"]]
res = res.rename(columns={"reasonRetired": "dnf_reason",
                          "pitStops": "pit_stops"})
res["status"].value_counts()

status
Finished    264
DNF          57
DNS           7
NC            2
Name: count, dtype: int64

## 5. Pit stops

F1DB times the whole pit lane visit (entry line to exit line),
not just the tyre change. So ~23 s is normal here, not ~2.5 s.

In [7]:
pit = add_names(pits)
pit["gp"] = pit["round"].map(gp_of)
pit["seconds"] = pit["timeMillis"] / 1000
pit = pit[["round", "gp", "driver", "code", "team",
           "stop", "lap", "seconds"]]
pit["seconds"].describe().round(2)

count    469.00
mean      24.76
std        5.08
min       17.08
25%       21.75
50%       23.32
75%       27.14
max       47.07
Name: seconds, dtype: float64

The long tail (40 s+) is worth a look before we plot it.

In [8]:
pit.nlargest(5, "seconds")

,round,gp,driver,code,team,stop,lap,seconds
22114,2,China,Esteban Ocon,OCO,Haas,2,46,47.067
22501,14,Spain,Carlos Sainz Jr.,SAI,Williams,1,23,46.367
22352,9,Great Britain,Lance Stroll,STR,Aston Martin,2,46,46.234
22308,9,Great Britain,Alexander Albon,ALB,Williams,1,1,44.179
22499,14,Spain,Fernando Alonso,ALO,Aston Martin,1,14,43.845


F1DB does not say why (likely penalties or repairs). We keep
them. The distribution charts in 03 show them as outliers rather
than dropping them.

## 6. Standings after every round

These include sprint points, which the race results table does not.
That is why standings, not race results, give the true totals.

In [9]:
ds = add_names(d_stand)[["round", "driver", "code",
                         "positionNumber", "points"]]
ds = ds.rename(columns={"positionNumber": "position"})

ts = t_stand.copy()
ts["team"] = ts["constructorId"].map(teams["name"])
ts = ts[["round", "team", "positionNumber", "points"]]
ts = ts.rename(columns={"positionNumber": "position"})

# driver's team as of their latest race, for coloring
latest = res.sort_values("round").groupby("driver")["team"].last()
ds["team"] = ds["driver"].map(latest)
ts[ts["round"] == LAST_ROUND].sort_values("position")

,round,team,position,points
10643,15,Mercedes,1.0,538.0
10644,15,Ferrari,2.0,378.0
10645,15,McLaren,3.0,306.0
10646,15,Red Bull,4.0,263.0
10647,15,Racing Bulls,5.0,83.0
10648,15,Alpine,6.0,68.0
10649,15,Haas,7.0,27.0
10650,15,Audi,8.0,17.0
10651,15,Williams,9.0,12.0
10652,15,Aston Martin,10.0,3.0


## 7. Calendar and circuits (for the map)

All 23 races, flagged raced or upcoming, with lat/long.
Round 16 is the Bahrain GP, but F1DB lists it at Sepang, Malaysia
("Bahrain Grand Prix in Malaysia"). We map it where it is held.

In [10]:
countries = raw("countries").set_index("id")
cal["circuit"] = cal["circuitId"].map(circuits["name"])
cal["lat"] = cal["circuitId"].map(circuits["latitude"])
cal["lon"] = cal["circuitId"].map(circuits["longitude"])
cid = cal["circuitId"].map(circuits["countryId"])
cal["country"] = cid.map(countries["name"])
cal["iso3"] = cid.map(countries["alpha3Code"])
cal["raced"] = cal["round"] <= LAST_ROUND

winners = res[res["finish"] == 1].set_index("round")["driver"]
cal["winner"] = cal["round"].map(winners)
cal = cal[["round", "gp", "date", "circuit", "country", "iso3",
           "lat", "lon", "raced", "winner", "circuitType"]]
cal.loc[cal["round"] == 16]

,round,gp,date,circuit,country,iso3,lat,lon,raced,winner,circuitType
1164,16,Bahrain,2026-10-04,Sepang,Malaysia,MYS,2.760556,101.7375,False,NaN,RACE


## 8. Team colors

F1DB has no colors. These hex codes are the official 2026 team
colors as published in F1's live timing feed (read via FastF1 in
our recon). 02 shows why we do not use them as-is.

In [11]:
team_colors = pd.DataFrame({
    "team": ["Mercedes", "Ferrari", "McLaren", "Red Bull",
             "Racing Bulls", "Alpine", "Haas", "Audi",
             "Williams", "Aston Martin", "Cadillac"],
    "official_hex": ["#00D7B6", "#ED1131", "#F47600", "#4781D7",
                     "#6C98FF", "#00A1E8", "#9C9FA2", "#F50537",
                     "#1868DB", "#229971", "#909090"],
})
assert set(team_colors["team"]) == set(res["team"])

## 9. Sanity checks

In [12]:
# 22 cars x 15 races
assert len(res) == 22 * LAST_ROUND
# every race has exactly one winner
assert res.groupby("round")["finish"].apply(
    lambda s: (s == 1).sum()).eq(1).all()
# classified finishers have a numeric position
assert res.loc[res["status"] == "Finished", "finish"].notna().all()
# team standings total = race points + sprint points
sprint = raw("races-sprint-race-results")
sprint = this_year(sprint)["points"].sum()
final = ts[ts["round"] == LAST_ROUND]["points"].sum()
assert final == res["points"].sum() + sprint
print("race pts", res["points"].sum(), "+ sprint", sprint,
      "=", final)

race pts 1515.0 + sprint 180.0 = 1695.0


## 10. Save

In [13]:
out = {"results": res, "pit_stops": pit,
       "driver_standings": ds, "team_standings": ts,
       "calendar": cal, "team_colors": team_colors}
for name, df in out.items():
    df.to_csv(CLEAN / f"{name}.csv", index=False)
    print(f"{name:17} {df.shape}")

results           (330, 12)
pit_stops         (469, 8)
driver_standings  (333, 6)
team_standings    (165, 4)
calendar          (23, 11)
team_colors       (11, 2)
